# Caracterización Lingüística para la Detección de Comentarios de Odio mediante NLP

**Autor:** Gregory Harutyunyan Gevorgyan | **Dominio:** NLP / Extracción de Características

**Resumen:**
Este proyecto desarrolla un pipeline de análisis de lenguaje natural (NLP) diseñado para extraer conclusiones estadísticas y morfológicas de un corpus masivo (>570k registros). El objetivo principal es identificar rasgos diferenciales entre el **discurso de odio** y el **discurso neutro**, facilitando la ingeniería de variables para modelos predictivos posteriores.

**Componentes del Pipeline:**

* **Procesamiento de Datos a Escala:** Implementación de muestreo estratificado y limpieza avanzada de texto (normalización Unicode y corrección de *mojibake*).
* **Análisis Morfosintáctico:** Uso de `es_core_news_md` de spaCy para etiquetado POS (Part-of-Speech) y análisis de género/número.
* **Extracción de Entidades (NER):** Identificación y comparativa de entidades nombradas (Localizaciones, Organizaciones, Personas) entre clases.
* **Visualización de Datos:** Generación de un dashboard de métricas para sintetizar hallazgos clave.

*Para ver detalle del conjunto de datos, consultar `DATA_DICTIONARY.md`.*

---

## Fase 1: Ingesta de Datos y Preprocesamiento

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">1.1. Ingesta y depuración del Dataset</span>


---


Importamos las librerias necesarias y cargamos el modelo de PLN.

---



In [1]:
import pathlib # Manejo de rutas y archivos de forma moderna
import spacy # Motor de NLP
import pandas as pd # Manejo de datasets

# Instalación en el entorno para poder cargarlo
!python -m spacy download es_core_news_md 

from spacy import displacy # Visualización de resultados
import csv # Leer archivos csv a bajo nivel sin usar pandas
import es_core_news_md # _sm/_md/_lg y la más precisa en contextos es es_dep_news_trf

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/42.3 MB ? eta -:--:--
     ---------------------------------------- 0.3/42.3 MB ? eta -:--:--
      --------------------------------------- 0.8/42.3 MB 2.2 MB/s eta 0:00:20
     - -------------------------------------- 1.3/42.3 MB 2.2 MB/s eta 0:00:19
     - -------------------------------------- 1.6/42.3 MB 2.2 MB/s eta 0:00:19
     - -------------------------------------- 2.1/42.3 MB 2.2 MB/s eta 0:00:18
     -- ------------------------------------- 2.6/42.3 MB 2.1 MB/s eta 0:00:19
     -- ------------------------------------- 3.1/42.3 MB 2.2 MB/s eta 0:00:18
     --- ------------------------------------ 3.4/42.3 MB 2.2 MB/s eta 0:00:18
     --- ------------------------------------ 3.9/42.3 MB 2.1 MB/s eta 0:00:18
     ---- ----------------------------------- 4.5/42.3 MB 2.1 MB/s eta 0:00:18
     ---- ----------------------------------- 4.7/42.3 MB 2.1 MB/s 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\grego\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Cargamos el modelo:
nlp = es_core_news_md.load()

In [3]:
## Conectamos con Google Drive y definimos la ruta del fichero
# from google.colab import drive
# drive.mount('/content/drive')

filename = "../data/raw/dataset.csv"

---
Cargamos el archivo CSV en memoria como un Dataframe de pandas con la codificación de caracteres latin-1.

---



In [4]:
# Leemos el fichero y contamos cuántas filas tiene:
def cont_filas_csv(filename):
  with open(filename, 'r', encoding='utf-8', errors='ignore') as file: # Abrimos el fichero de forma eficiente
    lector = csv.reader(file, delimiter=';') # Creamos un objeto lector de csv
    next(lector, None) # Saltamos la primera fila
    num_filas = sum(1 for filas in lector) # Sumamos las filas totales
  return num_filas

filas_csv = cont_filas_csv(filename)
print(f"Numero de filas en el csv: {filas_csv}")

Numero de filas en el csv: 575029


In [5]:
data = pd.read_csv(filename, delimiter=";", encoding='latin-1')

C:\Users\grego\AppData\Local\Temp\ipykernel_11868\3854135399.py:1: DtypeWarning: Columns (6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(filename, delimiter=";", encoding='latin-1')


In [6]:
print("Primeras filas del corpus original:")
data.head()

Primeras filas del corpus original:


,MEDIO,SOPORTE,URL,TIPO DE MENSAJE,CONTENIDO A ANALIZAR,INTENSIDAD,TIPO DE ODIO,TONO HUMORISTICO,MODIFICADOR,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,EL PAÍS,WEB,URL_a4d7efc0,COMENTARIO,el barí§a nunca acaeza ante un segundo b ni an...,3.0,Otros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EL PAÍS,WEB,URL_a4d7efc0,COMENTARIO,el real madrid ha puesto punto y final a su an...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EL PAÍS,WEB,URL_54312d9e,COMENTARIO,cristina cifuentes podrí­a haber sido la presi...,3.0,Ideológico,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EL PAÍS,WEB,URL_54312d9e,COMENTARIO,habrí­a que reabrir el caso. el supremo se ded...,3.0,Ideológico,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EL PAÍS,WEB,URL_54312d9e,COMENTARIO,me parece un poco exagerado pedir más de tres ...,3.0,Ideológico,Si,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
El dataset contiene columnas irrelevantes para esta actividad. Procederemos a eliminarlas para simplificar el coste computacional.


---




In [7]:
# Reducimos el dataset a las columnas que nos interesan
data_reducido = data.drop(data.columns[[0,1,2]], axis=1).drop(data.columns[7:], axis=1)

In [8]:
print("Primeras filas del corpus reducido:")
data_reducido.head(10)

Primeras filas del corpus reducido:


,TIPO DE MENSAJE,CONTENIDO A ANALIZAR,INTENSIDAD,TIPO DE ODIO
0,COMENTARIO,el barí§a nunca acaeza ante un segundo b ni an...,3.0,Otros
1,COMENTARIO,el real madrid ha puesto punto y final a su an...,0.0,NaN
2,COMENTARIO,cristina cifuentes podrí­a haber sido la presi...,3.0,Ideológico
3,COMENTARIO,habrí­a que reabrir el caso. el supremo se ded...,3.0,Ideológico
4,COMENTARIO,me parece un poco exagerado pedir más de tres ...,3.0,Ideológico
5,COMENTARIO,parece que todos los delincuentes niegan las e...,3.0,Ideológico
6,COMENTARIO,"preguntárselo al fracasado, media carrera en 6...",4.0,Ideológico
7,COMENTARIO,tenemos aí±os para ver a esta asociación delic...,3.0,Ideológico
8,COMENTARIO,"una ""fardera"" como presidenta de la comunidad ...",4.0,Ideológico
9,COMENTARIO,uno a uno tienen que ir cayendo frente a la ju...,3.0,Ideológico


### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">1.2. Limpieza del Dataset</span>



---

El corpus proporcionado contiene textos de distintas fuentes y presenta errores de codificación y ruido tipográfico. Para garantizar un análisis adecuado, se ha aplicado un proceso de limpieza y normalización.

---



In [9]:
import unicodedata
import re

reemplazos_concretos = {
    "í§": "ç",
    "í¼": "ü",
    "í±": "ñ",
    "í¡": "á",
    "í©": "é",
    "í³": "ó",
    "íº": "ú",
}

def correcion_especifica(texto):
    if not isinstance(texto, str):
        return texto
    for k, v in reemplazos_concretos.items():
        texto = texto.replace(k, v)
    return texto



def limpieza(texto):
    if not isinstance(texto, str):
        return texto

    # 1. Reparar mojibake UTF-8
    try:
        texto = texto.encode("latin1").decode("utf-8")
    except:
        pass

    # 2. Reparar corrupciones cp1252/latin-1
    texto = correcion_especifica(texto)

    # 3. Normalizar
    texto = unicodedata.normalize("NFKC", texto)

    # 4. Limpiar basura residual
    texto = re.sub(
        r"[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑçüÇÜ.,;:!?¿¡()\"' /-]+",
        "",
        texto
    )

    return texto

data_limpio = data_reducido.copy()

for col in data_limpio.columns:
    if data_limpio[col].dtype == object:
      data_limpio[col] = data_limpio[col].apply(limpieza)

In [10]:
# Verificamos que han habido cambios:
cambios = (data_reducido != data_limpio).sum()
cambios

TIPO DE MENSAJE            273
CONTENIDO A ANALIZAR    369059
INTENSIDAD                 399
TIPO DE ODIO            562736
dtype: int64

In [11]:
# Hallamos el número de filas de cada columna con valores faltantes
x = data_limpio["TIPO DE MENSAJE"].isna().sum()
y = data_limpio["CONTENIDO A ANALIZAR"].isna().sum()
z = data_limpio["INTENSIDAD"].isna().sum()

print(f"El numero de filas que contienen valores faltantes en la columna TIPO DE MENSAJE es: {x}")
print(f"El numero de filas que contienen valores faltantes en la columna CONTENIDO A ANALIZAR es: {y}")
print(f"El numero de filas que contienen valores faltantes en la columna INTENSIDAD es: {z}")

El numero de filas que contienen valores faltantes en la columna TIPO DE MENSAJE es: 262
El numero de filas que contienen valores faltantes en la columna CONTENIDO A ANALIZAR es: 268
El numero de filas que contienen valores faltantes en la columna INTENSIDAD es: 399


---
Debido a que el número de filas que contienen valores faltantes es insignificante en comparación al conjunto total de los datos, podemos eliminar esas filas sin problema.

---




In [12]:
# Limpiamos esas filas
data_limpio = data_limpio.dropna(subset=["INTENSIDAD"])

In [13]:
print("Primeras filas del dataset final limpio:")
data_limpio.head(10)

Primeras filas del dataset final limpio:


,TIPO DE MENSAJE,CONTENIDO A ANALIZAR,INTENSIDAD,TIPO DE ODIO
0,COMENTARIO,el barça nunca acaeza ante un segundo b ni ant...,3.0,Otros
1,COMENTARIO,el real madrid ha puesto punto y final a su an...,0.0,NaN
2,COMENTARIO,cristina cifuentes podría haber sido la presid...,3.0,Ideológico
3,COMENTARIO,habría que reabrir el caso. el supremo se dedi...,3.0,Ideológico
4,COMENTARIO,me parece un poco exagerado pedir más de tres ...,3.0,Ideológico
5,COMENTARIO,parece que todos los delincuentes niegan las e...,3.0,Ideológico
6,COMENTARIO,"preguntárselo al fracasado, media carrera en 6...",4.0,Ideológico
7,COMENTARIO,tenemos años para ver a esta asociación delict...,3.0,Ideológico
8,COMENTARIO,"una ""fardera"" como presidenta de la comunidad ...",4.0,Ideológico
9,COMENTARIO,uno a uno tienen que ir cayendo frente a la ju...,3.0,Ideológico


<hr>
Guardamos el dataset limpio en memoria local
<hr>

In [14]:
data_limpio.to_csv('../data/processed/dataset_cleaned.csv', index=False)

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">1.3. Submuestreo manteniendo las proporciones relevantes.</span>

---
Para la realización de la actividad supone mucho coste de tiempo usar el dataset en su totalidad. Por lo tanto, nos reduciremos a un subconjunto que mantenga las proporciones de las variables analizar para que la submuestra sea representativa.

---

In [15]:
lineas_submuestra = 40000

In [16]:
# Creamos un diccionario con el numero de TIPO DE MENSAJE teniendo en cuenta sin odio y con odio
def fun_num_mensajes(df):
  mensajes_sin_odio = df[df["INTENSIDAD"] == 0.0]["TIPO DE MENSAJE"].value_counts()
  mensajes_con_odio = df[df["INTENSIDAD"] != 0.0]["TIPO DE MENSAJE"].value_counts()

  tipo_mensaje = {
      "COMENTARIO": [int(mensajes_sin_odio.get("COMENTARIO",0)), int(mensajes_con_odio.get("COMENTARIO",0))],
      "NOTICIA": [int(mensajes_sin_odio.get("NOTICIA",0)), int(mensajes_con_odio.get("NOTICIA",0))],
      "TITULAR NOTICIA": [int(mensajes_sin_odio.get("TITULAR NOTICIA",0)), int(mensajes_con_odio.get("TITULAR NOTICIA",0))]
      }
  return tipo_mensaje

tipo_mensaje = fun_num_mensajes(data_limpio)

print("Número de cada TIPO DE MENSAJE:[sin_odio, con_odio]:")
tipo_mensaje

Número de cada TIPO DE MENSAJE:[sin_odio, con_odio]:


{'COMENTARIO': [323697, 10621],
 'NOTICIA': [157843, 1570],
 'TITULAR NOTICIA': [80793, 105]}

In [17]:
# Creamos una función que devuelva las proporciones de cada tipo de mensaje
def fun_proporciones(df,tipo_mensaje):
  num_lineas = df.shape[0]
  # Hallamos las proporciones de cada TIPO DE MENSAJE
  prop_comentario = [tipo_mensaje['COMENTARIO'][0] / num_lineas, tipo_mensaje['COMENTARIO'][1] / num_lineas]
  prop_noticia = [tipo_mensaje['NOTICIA'][0] / num_lineas, tipo_mensaje['NOTICIA'][1] / num_lineas]
  prop_titular = [tipo_mensaje['TITULAR NOTICIA'][0] / num_lineas, tipo_mensaje['TITULAR NOTICIA'][1] / num_lineas]

  # Creamos el diccionario de las proporciones
  proporciones = {
      "COMENTARIO": prop_comentario,
      "NOTICIA": prop_noticia,
      "TITULAR NOTICIA": prop_titular
      }
  return proporciones

proporciones = fun_proporciones(data_limpio, tipo_mensaje)

print("Proporciones de cada TIPO DE MENSAJE: [sin_odio, con_odio]:")
proporciones

Proporciones de cada TIPO DE MENSAJE: [sin_odio, con_odio]:


{'COMENTARIO': [0.5633137845222143, 0.018483197883855697],
 'NOTICIA': [0.27468631989280057, 0.002732192889337487],
 'TITULAR NOTICIA': [0.14060003828550546, 0.00018272627603849433]}

In [18]:
# Definimos la función para extraer una submuestra manteniendo las proporciones
def extrac_con_proporciones(df, filas, prop):
  # Calculamos el número de filas de cada tipo de mensaje:
  f_comentarios = [
      int(filas*prop["COMENTARIO"][0]),
      int(filas*prop["COMENTARIO"][1])
      ]
  f_noticia = [
      int(filas*prop["NOTICIA"][0]),
      int(filas*prop["NOTICIA"][1])
      ]
  f_titular = [
      int(filas*prop["TITULAR NOTICIA"][0]),
      int(filas*prop["TITULAR NOTICIA"][1])
      ]

  # Generamos las partes de la submuestra final del dataframe original
  df_comentarios_sin_odio = df[(df["TIPO DE MENSAJE"] == "COMENTARIO") & (df["INTENSIDAD"] == 0.0)].sample(n = f_comentarios[0], random_state = 8)
  df_comentarios_con_odio = df[(df["TIPO DE MENSAJE"] == "COMENTARIO") & (df["INTENSIDAD"] != 0.0)].sample(n = f_comentarios[1], random_state = 8)
  df_noticias_sin_odio = df[(df["TIPO DE MENSAJE"] == "NOTICIA") & (df["INTENSIDAD"] == 0.0)].sample(n = f_noticia[0], random_state = 8)
  df_noticias_con_odio = df[(df["TIPO DE MENSAJE"] == "NOTICIA") & (df["INTENSIDAD"] != 0.0)].sample(n = f_noticia[1], random_state = 8)
  df_titular_sin_odio = df[(df["TIPO DE MENSAJE"] == "TITULAR NOTICIA") & (df["INTENSIDAD"] == 0.0)].sample(n = f_titular[0], random_state = 8)
  df_titular_con_odio = df[(df["TIPO DE MENSAJE"] == "TITULAR NOTICIA") & (df["INTENSIDAD"] != 0.0)].sample(n = f_titular[1], random_state = 8)

  return pd.concat([df_comentarios_sin_odio,
                    df_comentarios_con_odio,
                    df_noticias_sin_odio,
                    df_noticias_con_odio,
                    df_titular_sin_odio,
                    df_titular_con_odio], ignore_index=True)

sub_data = extrac_con_proporciones(data_limpio, lineas_submuestra, proporciones)

sub_data.shape[0]


39998

---
Una vez extraída la submuestra comprobamos antes que mantiene las proporciones establecidas.

---

In [19]:
tipo_mensaje_sub = fun_num_mensajes(sub_data)

proporciones_sub = fun_proporciones(sub_data, tipo_mensaje_sub)

print("Proporciones de la submuestra de cada TIPO DE MENSAJE:[sin_odio, con_odio]:")
proporciones_sub


Proporciones de la submuestra de cada TIPO DE MENSAJE:[sin_odio, con_odio]:


{'COMENTARIO': [0.5633281664083204, 0.01847592379618981],
 'NOTICIA': [0.2746887344367218, 0.0027251362568128405],
 'TITULAR NOTICIA': [0.1406070303515176, 0.00017500875043752187]}

In [20]:
sub_data.head(10)

,TIPO DE MENSAJE,CONTENIDO A ANALIZAR,INTENSIDAD,TIPO DE ODIO
0,COMENTARIO,USUARIOOFUSCADO USUARIOOFUSCADO USUARIOOFUSCAD...,0.0,NaN
1,COMENTARIO,haciendo campaña publicitaria...se ve que este...,0.0,NaN
2,COMENTARIO,tambin había premio de literatura juvenil en c...,0.0,NaN
3,COMENTARIO,"sin embargo tu, terrazas a tope, bares a tope,...",0.0,NaN
4,COMENTARIO,"si me votáis, volveré a sacrificarme por vosot...",0.0,NaN
5,COMENTARIO,unidas podemos son unos analfabetos incapaces ...,0.0,NaN
6,COMENTARIO,en irak las mujeres ya pueden conducir coches ...,0.0,NaN
7,COMENTARIO,falta valentía en muchos votantes catalanes pa...,0.0,NaN
8,COMENTARIO,pues yo era de los que tenía ganas de poder ap...,0.0,NaN
9,COMENTARIO,pero ésto qué cachondeo es??,0.0,NaN


<hr>
Guardamos la submuestra del dataset limpio en local
<hr>

In [21]:
sub_data.to_csv('../data/processed/sub_dataset_cleaned.csv', index=False)